# LTX 2.3 22B — ComfyUI

LTX 2.3 ailesi: I2V, IC Control, ID LoRA, LipDub, Edit Anything, Face Swap, Dearchive, Movie Builder.

**Pipeline:** `Görsel/Prompt → LTX 2.3 22B → Upscale → Frame interpolation → MP4`

## Ön Hazırlık
- Cloudflare Dashboard: `comfy.ersamely.com` → `localhost:8188`
- Colab Secrets: `CF_TUNNEL_TOKEN`, `HF_TOKEN`

## Kullanım
A: Kurulum + Node'lar → B: Model indir → C: Başlat + Tunnel → Browser: `comfy.ersamely.com`

---
# A) Kurulum + Custom Node'lar

In [ ]:
import os
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

# ComfyUI
COMFY_DIR = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFY_DIR}/custom_nodes'

if not os.path.exists(COMFY_DIR):
    print('\U0001f4e6 ComfyUI...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
    !pip install -q -r {COMFY_DIR}/requirements.txt
else:
    print('\u2705 ComfyUI mevcut')

# ─── Custom Node'lar ───
NODES = {
    # Zorunlu
    'ComfyUI-LTXVideo': 'https://github.com/Lightricks/ComfyUI-LTXVideo.git',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    # Kalite artırıcı
    'ComfyUI-Frame-Interpolation': 'https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git',
    'ComfyUI-VideoUpscale_WithModel': 'https://github.com/ShmuelRonen/ComfyUI-VideoUpscale_WithModel.git',
    # Utility
    'ComfyUI-Manager': 'https://github.com/ltdrdata/ComfyUI-Manager.git',
    'ComfyUI-Impact-Pack': 'https://github.com/ltdrdata/ComfyUI-Impact-Pack.git',
    'rgthree-comfy': 'https://github.com/rgthree/rgthree-comfy.git',
    'ComfyUI-KJNodes': 'https://github.com/kijai/ComfyUI-KJNodes.git',
}

for name, url in NODES.items():
    node_dir = f'{CUSTOM_NODES}/{name}'
    if not os.path.exists(node_dir):
        print(f'  \u2193 {name}')
        !git clone --depth 1 {url} {node_dir}
        req_file = f'{node_dir}/requirements.txt'
        if os.path.exists(req_file):
            !pip install -q -r {req_file}
    else:
        print(f'  \u2713 {name}')

# kornia fix for LTXVideo
!pip install -q kornia==0.7.3

# Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('\n\u2705 Kurulum tamam')

---
# B) LTX 2.3 Model İndir

In [ ]:
import os
import glob

MODELS_DIR = f'{COMFY_DIR}/models'

# ╔══════════════════════════════════════════════════════════════╗
# ║  WORKFLOW SEÇ — değiştirip B'yi tekrar çalıştır            ║
# ║  Eski workflow modelleri otomatik silinir, ortak kalanlar   ║
# ╚══════════════════════════════════════════════════════════════╝
WORKFLOW = 'i2v'  # 'i2v' | 'ic_control' | 'id_lora' | 'lipdub' | 'edit_anything' | 'face_swap' | 'dearchive' | 'movie_builder'


def hf_download(repo, filename, dest_dir):
    basename = filename.split('/')[-1]
    dest = f'{dest_dir}/{basename}'
    if os.path.exists(dest):
        print(f'  \u2713 {basename} (mevcut)')
        return
    print(f'  \u2193 {basename}...')
    os.makedirs(dest_dir, exist_ok=True)
    from huggingface_hub import hf_hub_download
    try:
        path = hf_hub_download(repo_id=repo, filename=filename, local_dir='/content/hf_cache')
        import shutil
        shutil.move(path, dest)
        print(f'  \u2705 {basename}')
    except Exception as e:
        print(f'  \u274c Başarısız: {e}')


def remove_if_exists(path):
    if os.path.exists(path):
        os.remove(path)
        print(f'  \U0001f5d1 Silindi: {os.path.basename(path)}')


# ─── Ortak modeller (her workflow'da lazım, silinmez) ───
print('\U0001f4e5 Ortak modeller:')
hf_download('Comfy-Org/ltx-2', 'split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors', f'{MODELS_DIR}/text_encoders')
hf_download('Lightricks/LTX-2.3', 'ltx-2.3-spatial-upscaler-x2-1.1.safetensors', f'{MODELS_DIR}/latent_upscale_models')
hf_download('Kijai/LTX2.3_comfy', 'vae/LTX23_video_vae_bf16.safetensors', f'{MODELS_DIR}/vae')
hf_download('Comfy-Org/ltx-2.3', 'split_files/loras/ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors', f'{MODELS_DIR}/loras')

# ─── Workflow'a özel modeller ───
if WORKFLOW == 'i2v':
    print('\n\U0001f3ac Workflow: I2V (Image-to-Video)')
    # Önce eski modelleri sil
    remove_if_exists(f'{MODELS_DIR}/checkpoints/ltx-2.3-22b-distilled-fp8.safetensors')
    remove_if_exists(f'{MODELS_DIR}/geometry_estimation/moge_2_vitl_normal_fp16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-id-lora-talkvid-3k.safetensors')
    # Sonra indir
    hf_download('Lightricks/LTX-2.3-fp8', 'ltx-2.3-22b-dev-fp8.safetensors', f'{MODELS_DIR}/checkpoints')

elif WORKFLOW == 'ic_control':
    print('\n\U0001f3ac Workflow: IC-LoRA Union Control (V2V)')
    # Önce eski modelleri sil
    remove_if_exists(f'{MODELS_DIR}/checkpoints/ltx-2.3-22b-dev-fp8.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-id-lora-talkvid-3k.safetensors')
    # Sonra indir
    hf_download('Lightricks/LTX-2.3-fp8', 'ltx-2.3-22b-distilled-fp8.safetensors', f'{MODELS_DIR}/checkpoints')
    hf_download('Comfy-Org/MoGe', 'geometry_estimation/moge_2_vitl_normal_fp16.safetensors', f'{MODELS_DIR}/geometry_estimation')
    hf_download('Lightricks/LTX-2.3-22b-IC-LoRA-Union-Control', 'ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors', f'{MODELS_DIR}/loras')
    hf_download('Comfy-Org/ltx-2', 'split_files/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors', f'{MODELS_DIR}/loras')

elif WORKFLOW == 'id_lora':
    print('\n\U0001f3ac Workflow: ID LoRA (Talking Head)')
    # Önce eski modelleri sil
    remove_if_exists(f'{MODELS_DIR}/checkpoints/ltx-2.3-22b-distilled-fp8.safetensors')
    remove_if_exists(f'{MODELS_DIR}/geometry_estimation/moge_2_vitl_normal_fp16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors')
    # Sonra indir
    hf_download('Lightricks/LTX-2.3-fp8', 'ltx-2.3-22b-dev-fp8.safetensors', f'{MODELS_DIR}/checkpoints')
    hf_download('Comfy-Org/ltx-2.3', 'split_files/loras/ltx-2.3-id-lora-talkvid-3k.safetensors', f'{MODELS_DIR}/loras')

elif WORKFLOW == 'lipdub':
    print('\n\U0001f3ac Workflow: LipDub (Audio-to-Video)')
    # Önce eski modelleri sil
    remove_if_exists(f'{MODELS_DIR}/checkpoints/ltx-2.3-22b-dev-fp8.safetensors')
    remove_if_exists(f'{MODELS_DIR}/geometry_estimation/moge_2_vitl_normal_fp16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-id-lora-talkvid-3k.safetensors')
    # Sonra indir
    hf_download('Lightricks/LTX-2.3-fp8', 'ltx-2.3-22b-distilled-fp8.safetensors', f'{MODELS_DIR}/checkpoints')
    hf_download('Lightricks/LTX-2.3-22b-IC-LoRA-Union-Control', 'ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors', f'{MODELS_DIR}/loras')
    hf_download('Kijai/LTX2.3_comfy', 'vae/LTX23_audio_vae_bf16.safetensors', f'{MODELS_DIR}/vae')
    # Audio VAE symlink
    symlink_src = f'{MODELS_DIR}/vae/LTX23_audio_vae_bf16.safetensors'
    symlink_dst = f'{MODELS_DIR}/vae/ltx-2-3-22b-audio_vae.safetensors'
    if os.path.exists(symlink_src) and not os.path.exists(symlink_dst):
        os.symlink(symlink_src, symlink_dst)
        print(f'  \U0001f517 ltx-2-3-22b-audio_vae.safetensors → LTX23_audio_vae_bf16.safetensors')

elif WORKFLOW == 'edit_anything':
    print('\n\U0001f3ac Workflow: Edit Anything')
    # Önce eski modelleri sil
    remove_if_exists(f'{MODELS_DIR}/checkpoints/ltx-2.3-22b-distilled-fp8.safetensors')
    remove_if_exists(f'{MODELS_DIR}/geometry_estimation/moge_2_vitl_normal_fp16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-id-lora-talkvid-3k.safetensors')
    # Sonra indir
    hf_download('Lightricks/LTX-2.3-fp8', 'ltx-2.3-22b-dev-fp8.safetensors', f'{MODELS_DIR}/checkpoints')
    hf_download('Comfy-Org/ltx-2.3', 'split_files/loras/ltx-2.3-edit-anything-lora.safetensors', f'{MODELS_DIR}/loras')

elif WORKFLOW == 'face_swap':
    print('\n\U0001f3ac Workflow: Face Swap')
    # Önce eski modelleri sil
    remove_if_exists(f'{MODELS_DIR}/checkpoints/ltx-2.3-22b-dev-fp8.safetensors')
    remove_if_exists(f'{MODELS_DIR}/checkpoints/ltx-2.3-22b-distilled-fp8.safetensors')
    remove_if_exists(f'{MODELS_DIR}/geometry_estimation/moge_2_vitl_normal_fp16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-id-lora-talkvid-3k.safetensors')
    # Sonra indir
    hf_download('Comfy-Org/ltx-2.3', 'split_files/diffusion_models/ltx-2.3-22b-dev_transformer_only_fp8_scaled.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('Comfy-Org/ltx-2.3', 'split_files/loras/ltx-2.3-head-swap-lora.safetensors', f'{MODELS_DIR}/loras')

elif WORKFLOW == 'dearchive':
    print('\n\U0001f3ac Workflow: Dearchive')
    # Önce eski modelleri sil
    remove_if_exists(f'{MODELS_DIR}/checkpoints/ltx-2.3-22b-distilled-fp8.safetensors')
    remove_if_exists(f'{MODELS_DIR}/geometry_estimation/moge_2_vitl_normal_fp16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-id-lora-talkvid-3k.safetensors')
    # Sonra indir
    hf_download('Lightricks/LTX-2.3-fp8', 'ltx-2.3-22b-dev-fp8.safetensors', f'{MODELS_DIR}/checkpoints')
    hf_download('Comfy-Org/ltx-2.3', 'split_files/loras/ltx-2.3-dearchive-lora.safetensors', f'{MODELS_DIR}/loras')

elif WORKFLOW == 'movie_builder':
    print('\n\U0001f3ac Workflow: Movie Builder')
    # Önce eski modelleri sil
    remove_if_exists(f'{MODELS_DIR}/checkpoints/ltx-2.3-22b-distilled-fp8.safetensors')
    remove_if_exists(f'{MODELS_DIR}/geometry_estimation/moge_2_vitl_normal_fp16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/ltx-2.3-id-lora-talkvid-3k.safetensors')
    # Sonra indir
    hf_download('Lightricks/LTX-2.3-fp8', 'ltx-2.3-22b-dev-fp8.safetensors', f'{MODELS_DIR}/checkpoints')
    hf_download('Comfy-Org/flux2-klein-9B', 'split_files/diffusion_models/flux2-klein-9B-fp8.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('Comfy-Org/flux2-dev', 'split_files/vae/flux2-vae.safetensors', f'{MODELS_DIR}/vae')
    hf_download('Comfy-Org/flux2-klein-9B', 'split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors', f'{MODELS_DIR}/text_encoders')

print('\n\u2705 Model indirme tamam')

---
# C) ComfyUI Başlat

`USE_CLOUDFLARE = False` (default): Colab proxy — hızlı, URL değişir
`USE_CLOUDFLARE = True`: Cloudflare tunnel — yavaş ama sabit URL

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata, output

# ╔══════════════════════════════════════════════════╗
# ║  TUNNEL SEÇ                                    ║
# ╚══════════════════════════════════════════════════╝
USE_CLOUDFLARE = False  # False=Colab proxy (hızlı) | True=Cloudflare (sabit URL)

PORT = 8188

# Önceki process'leri kapat
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# ComfyUI başlat
log_file = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', str(PORT), '--gpu-only', '--enable-cors-header', '*'],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)
print(f'\U0001f680 ComfyUI başlatıldı (PID: {comfy_proc.pid})')

# Hazır olmasını bekle
t0 = time.time()
ready = False
while time.time() - t0 < 120:
    try:
        if requests.get(f'http://localhost:{PORT}/system_stats', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if comfy_proc.poll() is not None:
        print('\u274c ComfyUI çöktü!')
        log_file.close()
        with open('/content/comfyui.log') as f:
            print(f.read()[-500:])
        break
    time.sleep(3)

if ready:
    print(f'\u2705 ComfyUI hazır ({int(time.time()-t0)}s)')

    if USE_CLOUDFLARE:
        token = userdata.get('CF_TUNNEL_TOKEN')
        cf_log = open('/content/cloudflared.log', 'w')
        cf_proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
            stdout=cf_log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        )
        time.sleep(5)
        print(f'\U0001f310 Cloudflare: https://comfyui.ersamely.com')
    else:
        print(f'\U0001f310 Colab Proxy:')
        output.serve_kernel_port_as_window(PORT, path='/')
else:
    print('\u274c Timeout!')

In [ ]:
import time
from datetime import datetime, timezone

import requests

print('ComfyUI canlı tutma. Durdurmak için interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'http://localhost:{PORT}/system_stats', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False

    comfy_alive = comfy_proc.poll() is None
    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    c = '\u2705' if (local_ok and comfy_alive) else '\u274c'
    print(f'{now} | ComfyUI: {c}')
    time.sleep(30)